In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "resnet-50",
    "learning_rate": 0.001,
    "epochs": 10,
    "pretrained":True
}

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5],
                             [0.5, 0.5, 0.5])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5],
                             [0.5, 0.5, 0.5])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5],
                             [0.5, 0.5, 0.5])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:

resnet50 = models.resnet50(pretrained=True)

for param in resnet50.parameters():
    param.requires_grad = False

resnet50.fc = nn.Linear(resnet50.fc.in_features, 84)

for param in resnet50.fc.parameters():
    param.requires_grad = True

# for param in resnet50.layer4.parameters():
#     param.requires_grad = True

# for param in resnet50.parameters():
#     param.requires_grad = True

gpu = torch.device("cuda")
resnet50 = resnet50.to(gpu)


/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# for param in resnet50.layer4.parameters():
#     param.requires_grad = True

In [ ]:
for param in resnet50.parameters():
    param.requires_grad = True

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": resnet50.fc.parameters(), "lr": 1e-3},
    {"params": resnet50.layer4.parameters(), "lr": 1e-4},
    {"params": resnet50.layer1.parameters(), "lr": 1e-5}
])
epochs = model_config["epochs"]

In [7]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [8]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(resnet50, val_dataloader)

        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [14]:
train_model(resnet50, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.09263892312403448, Validation Accuracy: 0.923758972193739
Epoch: 2, Training Loss: 0.08461816169450533, Validation Accuracy: 0.923758972193739
Epoch: 3, Training Loss: 0.076274024033767, Validation Accuracy: 0.9229748476988962
Epoch: 4, Training Loss: 0.06826836068209069, Validation Accuracy: 0.9198986669883588
Epoch: 5, Training Loss: 0.06505653040995041, Validation Accuracy: 0.9208637432897039
Epoch: 6, Training Loss: 0.062291624153326446, Validation Accuracy: 0.9193558115688522
Epoch: 7, Training Loss: 0.055239079219220506, Validation Accuracy: 0.9212256469027083
Epoch: 8, Training Loss: 0.05420982590862709, Validation Accuracy: 0.9169431208154895
Epoch: 9, Training Loss: 0.050331772051860034, Validation Accuracy: 0.9240002412690753
Epoch: 10, Training Loss: 0.047136549211394686, Validation Accuracy: 0.9244827794197479


In [15]:
accuracy = validate_model(resnet50, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 92.07245636696807
